
# Appendix 03: Spherical Tables

Source span: printed pages 381-390; PDF pages 394-403. The source was inspected to identify the table families and their statistical purpose. This notebook does not copy printed table values, long text, screenshots, or page crops. It turns the spherical table appendix into executable calculators and visual diagnostics.

## Chapter Goal

The spherical appendix serves the same role for `S^2` that the circular tables serve for the circle: it provides quantiles, concentration lookups, and reference thresholds for spherical inference. A standalone course notebook should expose the geometry behind those values. A Fisher colatitude quantile is a cap radius. A Fisher concentration estimate solves an equation for mean resultant length. A Watson axial concentration function describes how mass concentrates around a pole or around an equatorial girdle. A two-sample critical value is a reference distribution generated under a model.

The notebook builds four executable replacements. First, it draws Fisher cap quantiles and concentration inversion. Second, it visualizes how Fisher and Watson models place mass on a sphere. Third, it simulates resultant critical values under spherical uniformity and a two-sample equal-mean null. Fourth, it saves an interactive surface showing how sample size and significance level affect the Rayleigh-style uniformity threshold.

The point is not to reproduce every row of the printed appendix. The point is to teach how such rows are produced, what assumptions they encode, and which invariants must be checked before a table value is trusted.



## Visualization Storyboard And Library Routing

- **Fisher quantile and inversion atlas.** A Fisher distribution on `S^2` concentrates around a pole. Its colatitude quantile is literally the angular radius of a probability cap. Matplotlib is used for durable curves: cap half-angles, `A_3(kappa)`, and inverse concentration lookup.

- **Model-density sphere panels.** Spherical tables are easier to read when the reader can see the pole, cap, and girdle. A Matplotlib 3D surface colors the sphere by Fisher, Watson bipolar, and Watson girdle densities. This gives a quick visual distinction between directional and axial concentration.

- **Reference-distribution simulator.** Uniformity and two-sample spherical thresholds are generated by simulation. NumPy produces uniform sphere samples; SciPy supplies quantiles; the notebook records Monte Carlo settings and quantile ordering.

- **Interactive critical surface.** Plotly is used for one interactive artifact: the upper critical value of a spherical Rayleigh statistic as a function of sample size and alpha. This exposes the row/column interpolation problem that printed tables hide.

Every visual is paired with a numerical check: monotonicity, inverse round-trip residuals, density positivity, unit-vector norms, quantile ordering, or artifact existence.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "appendix-03"
source_span = {"printed": "381-390", "pdf": "394-403"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import optimize, special

from utils.artifacts import display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

rng = np.random.default_rng(3003)
np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def sample_uniform_sphere(count, *, rng=rng):
    z = rng.uniform(-1.0, 1.0, size=count)
    phi = rng.uniform(0.0, 2.0 * np.pi, size=count)
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(phi), r * np.sin(phi), z])


def sample_vmf_pole(kappa, count, *, rng=rng):
    if kappa <= 1e-12:
        return sample_uniform_sphere(count, rng=rng)
    u = rng.uniform(size=count)
    z = (1.0 / kappa) * np.log(np.exp(-kappa) + u * (np.exp(kappa) - np.exp(-kappa)))
    phi = rng.uniform(0.0, 2.0 * np.pi, size=count)
    r = np.sqrt(np.maximum(0.0, 1.0 - z * z))
    return np.column_stack([r * np.cos(phi), r * np.sin(phi), z])


def fisher_A3(kappa):
    kappa = np.asarray(kappa, dtype=float)
    out = np.zeros_like(kappa)
    mask = np.abs(kappa) > 1e-8
    out[mask] = 1.0 / np.tanh(kappa[mask]) - 1.0 / kappa[mask]
    out[~mask] = kappa[~mask] / 3.0
    return out if out.shape else float(out)


def fisher_A3_inverse(resultant):
    if resultant <= 1e-12:
        return 0.0
    if not 0.0 < resultant < 1.0:
        raise ValueError("resultant must lie in (0, 1)")
    upper = max(8.0, 1.0 / max(1e-4, 1.0 - resultant))
    f = lambda value: fisher_A3(value) - resultant
    while f(upper) < 0:
        upper *= 2.0
    return float(optimize.brentq(f, 1e-10, upper, maxiter=100))


def fisher_colatitude_quantile(kappa, probability):
    if kappa <= 1e-12:
        return np.arccos(1.0 - 2.0 * probability)
    numerator = np.exp(kappa) - probability * (np.exp(kappa) - np.exp(-kappa))
    cos_theta = np.log(numerator) / kappa
    return float(np.arccos(np.clip(cos_theta, -1.0, 1.0)))


def kummer_D3(kappa):
    kappa = np.asarray(kappa, dtype=float)
    return (1.0 / 3.0) * special.hyp1f1(1.5, 2.5, kappa) / special.hyp1f1(0.5, 1.5, kappa)


def resultant_length(points, axis=-2):
    mean = np.mean(points, axis=axis)
    return np.linalg.norm(mean, axis=-1)


def rayleigh_sphere_stat(points):
    n = points.shape[-2]
    r = resultant_length(points, axis=-2)
    return 3.0 * n * r * r


def two_sample_mean_stat(x, y):
    nx = x.shape[-2]
    ny = y.shape[-2]
    mx = np.mean(x, axis=-2)
    my = np.mean(y, axis=-2)
    return (nx * ny / (nx + ny)) * np.sum((mx - my) ** 2, axis=-1)



## Fisher Cap Quantiles And Concentration Lookup

For a Fisher distribution centered at the north pole, a colatitude quantile is a spherical cap radius. A 90 percent quantile of 20 degrees means that 90 percent of the probability mass lies inside a cap of angular radius 20 degrees around the pole. This is geometrically clearer than treating the quantile as just another row in a table.

The same distribution uses the function `A_3(kappa)=coth(kappa)-1/kappa` to connect concentration with mean resultant length on `S^2`. The inverse function is the spherical concentration lookup: observe `Rbar`, solve for `kappa`, and then use that estimate in inference. The dashboard below shows both jobs and checks that the inverse is stable on the plotted range.


In [ ]:

kappa_grid = np.linspace(0.001, 15.0, 500)
prob_levels = [0.50, 0.80, 0.90, 0.95]
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5))

ax = axes[0, 0]
for q in prob_levels:
    quantiles = [np.degrees(fisher_colatitude_quantile(k, q)) for k in kappa_grid]
    ax.plot(kappa_grid, quantiles, label=f"{int(100*q)}% cap")
ax.set_title("Fisher colatitude quantiles as cap radii")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("colatitude in degrees")
ax.legend()

ax = axes[0, 1]
ax.plot(kappa_grid, fisher_A3(kappa_grid), color="#345995")
ax.set_title("A_3(kappa)=coth(kappa)-1/kappa")
ax.set_xlabel("concentration kappa")
ax.set_ylabel("mean resultant length")
ax.set_ylim(0, 1.02)

ax = axes[1, 0]
r_grid = np.linspace(0.05, 0.97, 28)
inv_grid = np.array([fisher_A3_inverse(r) for r in r_grid])
ax.plot(r_grid, inv_grid, marker="o", color="#E07A5F")
ax.set_title("Executable Fisher inverse concentration table")
ax.set_xlabel("observed Rbar")
ax.set_ylabel("estimated kappa")

ax = axes[1, 1]
for kappa in [0.5, 1.5, 4.0, 9.0]:
    theta = np.linspace(0, np.pi, 500)
    density = (kappa / (2.0 * np.sinh(kappa))) * np.exp(kappa * np.cos(theta)) * np.sin(theta)
    ax.plot(np.degrees(theta), density, label=f"kappa={kappa:g}")
ax.set_title("Colatitude density on S^2")
ax.set_xlabel("colatitude in degrees")
ax.set_ylabel("density")
ax.legend()

fig.suptitle("Appendix 03 Fisher table replacements: caps, A_3, inverse lookup, and colatitude density", y=1.02, fontsize=14)
fig.tight_layout()
fisher_path = save_matplotlib(fig, TOPIC, "fisher-tables", "fisher-cap-quantile-and-inversion-dashboard.png")
plt.close(fig)
display_artifact(fisher_path, width=920)

fisher_diagnostics = {
    "A3_monotone": bool(np.all(np.diff(fisher_A3(kappa_grid)) > 0)),
    "A3_inverse_roundtrip_max_abs_error": float(max(abs(fisher_A3(fisher_A3_inverse(r)) - r) for r in r_grid)),
    "cap_radii_decrease_with_concentration": bool(
        all(np.all(np.diff([fisher_colatitude_quantile(k, q) for k in [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]]) < 0) for q in prob_levels)
    ),
}
fisher_diagnostics



## Fisher Versus Watson Geometry On The Sphere

Fisher models are directional: a pole and its antipode are different. Watson models are axial: the axis matters but the sign can be irrelevant. Positive axial concentration creates a bipolar pattern near the two poles; negative axial concentration creates a girdle near the equator. Spherical tables for Fisher and Watson models should therefore not be read as interchangeable concentration tables.

The next artifact colors the same sphere by three model shapes. The Fisher panel has one bright cap. The Watson bipolar panel has two antipodal bright caps. The Watson girdle panel has a bright belt. The diagnostic check confirms that the axial moment ratio `D_3(kappa)` stays between zero and one and increases with `kappa`, which is the moment equation behind the Watson concentration lookup.


In [ ]:

phi = np.linspace(0, 2 * np.pi, 96)
theta = np.linspace(0, np.pi, 48)
Phi, Theta = np.meshgrid(phi, theta)
X = np.sin(Theta) * np.cos(Phi)
Y = np.sin(Theta) * np.sin(Phi)
Zs = np.cos(Theta)

fisher_density = np.exp(4.0 * Zs)
watson_bipolar = np.exp(5.0 * Zs**2)
watson_girdle = np.exp(-5.0 * Zs**2)
panels = [(fisher_density, "Fisher pole"), (watson_bipolar, "Watson bipolar axis"), (watson_girdle, "Watson girdle")]

fig = plt.figure(figsize=(12, 4.5))
for idx, (density, title) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, idx, projection="3d")
    scaled = (density - density.min()) / (density.max() - density.min())
    facecolors = plt.cm.magma(scaled)
    ax.plot_surface(X, Y, Zs, facecolors=facecolors, linewidth=0, antialiased=False, shade=False)
    ax.plot([0, 0], [0, 0], [-1.15, 1.15], color="black", linewidth=1.0)
    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=24, azim=35)
fig.suptitle("Spherical table geometry: directional cap, axial poles, and girdle", y=1.02, fontsize=14)
fig.tight_layout()
model_path = save_matplotlib(fig, TOPIC, "model-geometry", "fisher-watson-sphere-density-panels.png")
plt.close(fig)
display_artifact(model_path, width=920)

k_watson = np.linspace(-6.0, 8.0, 500)
D_values = kummer_D3(k_watson)
watson_diagnostics = {
    "D3_values_finite": bool(np.all(np.isfinite(D_values))),
    "D3_values_between_zero_and_one": bool(np.all((D_values > 0) & (D_values < 1))),
    "D3_increases_with_kappa": bool(np.all(np.diff(D_values) > -1e-8)),
    "D3_at_zero_near_one_third": float(abs(kummer_D3(np.array([0.0]))[0] - 1.0 / 3.0)),
}
watson_diagnostics



## Spherical Uniformity And Two-Sample Critical Values

The spherical appendix also contains reference thresholds for tests. The key idea is the same as in the circular appendix: the table is a statement about a statistic under a model. For uniformity, a Rayleigh-style statistic `3 n Rbar^2` measures whether the sample has a preferred direction. For a two-sample equal-mean check, the distance between sample mean vectors is compared with its null distribution.

The simulation below is intentionally transparent. It generates unit vectors uniformly on the sphere, computes the statistics, and stores upper quantiles. The plot shows how sample size and alpha change the critical values. This is a table replacement with the model assumptions visible in code.


In [ ]:

n_values = [8, 12, 20, 40]
alpha_levels = [0.10, 0.05, 0.01]
reps = 6000
critical_rows = []
for n in n_values:
    samples = sample_uniform_sphere(reps * n).reshape(reps, n, 3)
    rayleigh = rayleigh_sphere_stat(samples)
    x = sample_uniform_sphere(reps * n).reshape(reps, n, 3)
    y = sample_uniform_sphere(reps * n).reshape(reps, n, 3)
    two_sample = two_sample_mean_stat(x, y)
    for alpha in alpha_levels:
        critical_rows.append({
            "n": n,
            "alpha": alpha,
            "rayleigh_upper": float(np.quantile(rayleigh, 1.0 - alpha)),
            "two_sample_upper": float(np.quantile(two_sample, 1.0 - alpha)),
        })

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for alpha in alpha_levels:
    axes[0].plot(n_values, [row["rayleigh_upper"] for row in critical_rows if row["alpha"] == alpha], marker="o", label=f"alpha={alpha:g}")
    axes[1].plot(n_values, [row["two_sample_upper"] for row in critical_rows if row["alpha"] == alpha], marker="o", label=f"alpha={alpha:g}")
axes[0].set_title("Spherical Rayleigh uniformity thresholds")
axes[0].set_xlabel("sample size n")
axes[0].set_ylabel("upper critical value")
axes[1].set_title("Two-sample mean-vector null thresholds")
axes[1].set_xlabel("sample size per group")
axes[1].set_ylabel("upper critical value")
for ax in axes:
    ax.legend()
fig.suptitle("Monte Carlo spherical critical-value table replacements", y=1.04, fontsize=14)
fig.tight_layout()
critical_path = save_matplotlib(fig, TOPIC, "spherical-critical-values", "spherical-critical-value-simulator.png")
plt.close(fig)
display_artifact(critical_path, width=920)

ordering_ok = True
for n in n_values:
    ray_vals = [row["rayleigh_upper"] for row in critical_rows if row["n"] == n]
    two_vals = [row["two_sample_upper"] for row in critical_rows if row["n"] == n]
    ordering_ok = ordering_ok and (ray_vals[0] < ray_vals[1] < ray_vals[2]) and (two_vals[0] < two_vals[1] < two_vals[2])

critical_diagnostics = {
    "uniform_sphere_unit_norm_max_error": float(np.max(np.abs(np.linalg.norm(samples.reshape(-1, 3), axis=1) - 1.0))),
    "critical_quantile_ordering_ok": bool(ordering_ok),
    "rayleigh_mean_near_chi_square_three": float(abs(np.mean(rayleigh) - 3.0)),
    "critical_rows": len(critical_rows),
}
critical_diagnostics



## Interactive Rayleigh Surface

The printed spherical tables have a grid structure: sample sizes in one direction, probabilities or significance levels in another. The interactive surface shows that structure for the spherical Rayleigh statistic. It is a compact way to see why interpolation between table entries is not just clerical. The statistic stabilizes toward its large-sample reference, but finite-sample movement is still visible for small `n`.


In [ ]:

surface_n = np.array([6, 8, 10, 12, 16, 20, 30, 40, 60])
surface_alpha = np.array([0.20, 0.10, 0.05, 0.025, 0.01])
Zcrit = []
for alpha in surface_alpha:
    row = []
    for n in surface_n:
        samples_surface = sample_uniform_sphere(4500 * n).reshape(4500, n, 3)
        row.append(float(np.quantile(rayleigh_sphere_stat(samples_surface), 1.0 - alpha)))
    Zcrit.append(row)
Zcrit = np.array(Zcrit)
N, A = np.meshgrid(surface_n, surface_alpha)
fig_surface = go.Figure(data=[go.Surface(x=N, y=A, z=Zcrit, colorscale="Viridis")])
fig_surface.update_layout(
    title="Spherical Rayleigh critical-value surface under uniformity",
    scene=dict(
        xaxis_title="sample size n",
        yaxis_title="alpha",
        zaxis_title="upper critical value",
        yaxis=dict(autorange="reversed"),
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=620,
)
surface_path = save_plotly_html(fig_surface, TOPIC, "interactive", "spherical-rayleigh-critical-surface.html", include_plotlyjs=True)
display_artifact(surface_path, width="100%", height=640)

surface_diagnostics = {
    "surface_shape": list(Zcrit.shape),
    "surface_values_positive": bool(np.all(Zcrit > 0)),
    "surface_critical_values_increase_as_alpha_decreases": bool(np.all(np.diff(Zcrit, axis=0) > 0)),
}
surface_diagnostics



## How To Use These Spherical Tables

A spherical lookup value is meaningful only after the geometry and model are named. A Fisher cap quantile assumes a directional distribution around a pole. A Fisher concentration estimate solves a mean-resultant equation on `S^2`. A Watson concentration function is axial and therefore treats antipodal directions differently from a Fisher model. A uniformity threshold assumes independent uniform points on the sphere.

The executable version makes those assumptions visible. If a chapter needs a cap radius, it can compute the Fisher quantile directly. If it needs concentration, it can solve `A_3(kappa)=Rbar` and inspect the round-trip residual. If it needs a critical value, it can simulate the null with a recorded seed, sample size, and alpha. The notebook therefore converts the appendix from a static lookup list into a reproducible reference lab.


In [ ]:

numeric_checks = {
    **fisher_diagnostics,
    **watson_diagnostics,
    **critical_diagnostics,
    **surface_diagnostics,
    "source_span": source_span,
    "critical_table_preview": critical_rows[:8],
    "all_boolean_checks_pass": bool(
        all(value for value in fisher_diagnostics.values() if isinstance(value, bool))
        and all(value for value in watson_diagnostics.values() if isinstance(value, bool))
        and all(value for value in critical_diagnostics.values() if isinstance(value, bool))
        and all(value for value in surface_diagnostics.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert numeric_checks["A3_inverse_roundtrip_max_abs_error"] < 1e-9
assert numeric_checks["rayleigh_mean_near_chi_square_three"] < 0.18
checks_path = save_json(numeric_checks, TOPIC, "checks", "spherical-table-replacement-checks.json")
display_artifact(checks_path)
numeric_checks


## Standalone Reading Notes

This appendix is meant to be used beside the spherical inference chapters as an executable reference. When a later calculation asks for a table value, first identify whether the value comes from a Fisher cap probability, an inverse concentration equation, an axial Watson moment, or a simulated null distribution. Those four jobs have different geometry. A cap radius depends on a chosen pole; a concentration estimate depends on mean resultant length; a Watson lookup treats antipodal directions as part of the same axis; a critical value depends on the statistic and the assumed null.

The generated simulations are intentionally modest and transparent. They demonstrate how a table row can be rebuilt with a seed, a sample size, a statistic, and a quantile. For production numerical work, increase the repetition count and record Monte Carlo error, but do not change the conceptual contract: samples must stay on the unit sphere, the statistic must match the test, and the saved residuals should make the lookup auditable.



## Takeaways

Appendix 03 is an executable reference for spherical inference.

- Fisher colatitude quantiles are cap radii. Drawing them keeps the probability statement attached to the sphere.
- The spherical concentration lookup is the inverse of `A_3(kappa)`, and the round-trip residual is the simplest audit of the calculation.
- Watson tables are axial. Positive and negative axial concentration produce visually different pole and girdle patterns, so they should not be flattened into generic concentration numbers.
- Spherical critical values are model-based simulation outputs. A reproducible notebook can state the null, generate unit vectors, compute the statistic, and record the quantile ordering.

These habits let later spherical chapters cite a table without losing the geometry that created it.


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [fisher_path, model_path, critical_path, surface_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "A3_monotone": fisher_diagnostics["A3_monotone"],
        "A3_inverse_roundtrip_under_tolerance": fisher_diagnostics["A3_inverse_roundtrip_max_abs_error"] < 1e-9,
        "Watson_D3_bounds_valid": watson_diagnostics["D3_values_between_zero_and_one"],
        "spherical_critical_quantiles_ordered": critical_diagnostics["critical_quantile_ordering_ok"],
        "interactive_surface_monotone_in_alpha": surface_diagnostics["surface_critical_values_increase_as_alpha_decreases"],
    },
    "pdf_used_for": "source orientation only; no copied tables, charts, screenshots, or page crops",
    "standalone_contract": "spherical table calculators, model-geometry visuals, simulation diagnostics, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 100
final_sanity
